# Задание 1. Модель для продолжения текста
Начнём с задачи предсказания следующего токена. Для этого будем использовать модель `distilgpt2` — языковую модель, предобученную для разных задач генерации текста. 

In [1]:
import joblib
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [2]:
load_model_distilgpt2 = False
model_name = "distilgpt2"          # лёгкая версия GPT-2

if load_model_distilgpt2:
    # 1) Загрузка токенизатора и модели
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    joblib.dump(model, '../models/' + model_name + '_model_09.joblib')
    joblib.dump(tokenizer, '../models/' + model_name + '_tokenizer_09.joblib')
else:
    model = joblib.load('../models/' + model_name + '_model_09.joblib')
    tokenizer = joblib.load('../models/' + model_name + '_tokenizer_09.joblib')

In [16]:
# 2) Создаём pipeline для генерации
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1  # -1 = CPU; 0 = первый GPU (если есть)
)

# 3) Исходный промпт
prompt = "when the AI shall be really creative"   # если по-русски, то токенайзер с русского все испортит

# 4) Генерируем продолжение
out = generator(
    prompt,
    max_new_tokens=10,   # пришлось добавить
    max_length=80,       # итоговая длина (включая prompt)
    num_return_sequences=1,
    do_sample=True,      # стохастическая генерация
    top_p=0.95,          # nucleus sampling
    temperature=0.8
)

print('='*80)
print(out[0]["generated_text"]) 

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=10) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


when the AI shall be really creative, but at the same time he will be going


Как видите, GPT можно запускать даже у себя на компьютере :
Поговорим о параметрах генерации подробнее. Вы могли заметить, что при вызове пайплайна генерации мы использовали не только prompt, но и другие параметры. Вот  они:
* `max_length` — задаёт максимальное количество новых токенов, которые мы можем сгенерировать. При достижении этой длины генерация останавливается.
* `num_return_sequences` — задаёт количество независимых ответов, которые можно сгенерировать на каждый запрос в батче.
* `do_sample` — определяет стратегию генерации (детерминистскую, когда берётся только наиболее вероятный токен, или какую-то другую). Если False, то на каждом шаге генерации будет выбираться только наиболее вероятный токен. Если True, то включим семплирование.
* `top_p` — настраивает beam search-генерацию, при которой генерируется сразу несколько цепочек токенов, а затем выбирается наиболее вероятная. Этот параметр ограничивает число слов, из которых модель случайным образом выбирает следующее слово в генерации текста. Это помогает создавать ответы с разными вариантами слов. Значение по умолчанию 0.9, но если хочется рассмотреть больше вариантов, его можно повысить до 0.95.
* `temperature` — температура в softmax-распределении. Если параметр очень большой, то распределение становится практически равномерным. Если же близок к нулю, то распределение вырождается в argmax. Стандартным значением считается единица. Чтобы сделать предсказания чуть более детерминистичными, этот параметр можно понизить, а если хочется больше разнообразия — повысить. Но сильно отходить от стандартных значений не советуем.

На самом деле параметров генерации гораздо больше: читайте подробнее обо всех параметрах, доступных в библиотеке transformers, и сценариях генерации.

# Задание 2. Модель для ответа на вопросы
Для генерации ответа на вопросы возьмём другую модель, специально обученную для этого. 
Ниже в переменной context записан текст. 

# Задание 2. Модель для ответа на вопросы
Для генерации ответа на вопросы возьмём другую модель, специально обученную для этого. 
* Ниже в переменной context записан текст. 

In [18]:
# 1) Загружаем pipeline для вопросно-ответной задачи
qa_pipeline = pipeline(
    task="question-answering",
    model="distilbert-base-uncased-distilled-squad",
    device=-1   # CPU
)

# 2) Пример входных данных
context = """
Transformers are neural network architectures introduced in 2017 by Vaswani et al. 
They rely entirely on self-attention mechanisms and have revolutionized natural language processing.
"""

question = "Who introduced the Transformer architecture?"

# 3) Получаем ответ
result = qa_pipeline({
    "context": context,
    "question": question
})

print(f"Answer: {result['answer']}") 

config.json:   0%|          | 0.00/451 [00:00<?, ?B/s]

c:\Users\aseva\Desktop\MyEDU\YaDLE\YaDLE_project_vscode_stream_2\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aseva\.cache\huggingface\hub\models--distilbert-base-uncased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' pack

model.safetensors:   0%|          | 0.00/265M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Answer: Vaswani et al


c:\Users\aseva\Desktop\MyEDU\YaDLE\YaDLE_project_vscode_stream_2\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(


# Задание 3. Модель для классификации текста
В мире глубокого обучения есть много широко известных задач классификации, и для них тоже есть предобученные модели. Напишите ниже ваше мнение о каком-нибудь фильме (на русском языке), а модель решит, позитивный ваш отзыв или негативный

In [19]:
classifier = pipeline(
    task="text-classification",
    model="cointegrated/rubert-tiny-sentiment-balanced",
    tokenizer="cointegrated/rubert-tiny-sentiment-balanced",
    device=-1
)

text = "Этот фильм был неожиданно интересным и очень трогательным."

result = classifier(text)

print(f"Метка: {result[0]['label']}, Уверенность: {result[0]['score']:.4f}")

config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

c:\Users\aseva\Desktop\MyEDU\YaDLE\YaDLE_project_vscode_stream_2\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aseva\.cache\huggingface\hub\models--cointegrated--rubert-tiny-sentiment-balanced. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet'

model.safetensors:   0%|          | 0.00/47.2M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/377 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


Метка: positive, Уверенность: 0.6400
